In [ ]:
!mineru -p ./pdfs/energies-18-00645.pdf -o ./mineru/mineru_rag_output -b pipeline

2026-07-21 15:58:26.902 | INFO     | mineru.cli.client:run_orchestrated_cli:953 - Started local mineru-api at http://127.0.0.1:59905
2026-07-21 15:58:33.804 | INFO     | __main__:create_app:236 - Request concurrency limited to 3
Start MinerU FastAPI Service: http://127.0.0.1:59905
API documentation: http://127.0.0.1:59905/docs
INFO:     Started server process [299070]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:59905 (Press CTRL+C to quit)
2026-07-21 15:58:33.949 | INFO     | mineru.cli.client:run_planned_task:832 - Submitting batch 1/1 | 1 document, 33 pages in this batch | 33 pages total | task#1 [energies-18-00645]
2026-07-21 15:58:40.932 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze_streaming:209 - Pipeline processing-window multi-file run. doc_count=1, total_pages=33, window_size=64, total_batches=1
2026-07-21 15:58:54.231 | INFO     | mineru.backend.pipeline.pipeline_analyze:d

In [ ]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import re

embeddings_model = OllamaEmbeddings(model="qwen3-embedding:4b")

def construct_vectorstore():
    
    text_splitter = RecursiveCharacterTextSplitter(
      
        chunk_size=2000,      
        chunk_overlap=150     
    )

    markdown_path = "./mineru/mineru_rag_output/energies-18-00645/auto/energies-18-00645.md"
    with open(markdown_path, "r", encoding="utf-8") as f:
        markdown_text = f.read()

    heading_pattern = re.compile(
        r"(?m)^(?P<heading>(?:#{1,6}\s*)?\d+(?:\.\d+)*\.\s+.+)$"
    )

    sections = []
    last_end = 0
    current_heading = None
    for match in heading_pattern.finditer(markdown_text):
        if current_heading is not None:
            body = markdown_text[last_end:match.start()].strip()
            if body:
                sections.append(Document(
                    page_content=f"{current_heading}\n{body}",
                    metadata={"heading": current_heading}
                ))
        current_heading = match.group("heading").strip()
        last_end = match.end()

    if current_heading is not None:
        body = markdown_text[last_end:].strip()
        if body:
            sections.append(Document(
                page_content=f"{current_heading}\n{body}",
                metadata={"heading": current_heading}
            ))

    if not sections:
        print(markdown_text)
        sections = [Document(page_content=markdown_text, metadata={})]

    token_chunks = []
    for section in sections:
        token_chunks.extend(text_splitter.split_documents([section]))
    
    vector_store = Chroma.from_documents(
        documents=token_chunks, embedding=embeddings_model, persist_directory="./mineru/mineru_rag_chroma")
    
    return vector_store


construct_vectorstore()

/home/zbta138a/miniconda3/envs/MinerU/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
vectordb = Chroma(
    persist_directory="./mineru/mineru_rag_chroma", embedding_function=embeddings_model
)
print(vectordb.get())

{'ids': ['187cad2e-1872-4c56-b562-086f65c3ce47', '1d6ab4f7-3214-484e-8db1-5aaa647afb14', '447bcdd0-aa3f-43c5-bcb5-f2b8942c93e7', '07980f26-ceb8-4988-a736-cb15a2191b22', 'cf887c63-6295-48b5-8342-e71adf54057c', 'dfbad847-b598-4ca7-b0e5-2eeb6685383d', '4294d6c7-e2ca-4ee8-b09c-076e4892d7eb', '17b4dc45-70b5-4ceb-949a-a221a27620e0', '455c3485-73e2-4378-9c24-d66524f01617', '2eea6954-dcd8-4b4b-9d8c-2befb3d21fd5', '7d021a51-0ef5-48fe-abb9-4349d66a51ad', '5705a9c1-c0e3-462c-aacd-67e41c9fe0b9', 'fb9643e9-0d3e-4efe-88cb-799547d9b72e', '8af3f022-185a-4732-9810-f400dc3106b4', 'eacdc980-8b18-4404-bdef-ab168488a9b0', '5e719824-197d-49db-ae4c-865ea855a84e', '0b49aefb-c411-4c1b-ac2e-36164fab63b8', '5d457c59-fd84-4e90-990c-c11bca2e8678', 'c071e5fb-f73f-4de1-8d54-cb0764f4515f', 'e90c60be-f32c-44b7-b395-86110555501f', 'f5bd3a39-59f4-4974-b60e-1064b6330845', 'c1865c4c-93ab-4c5f-8bf8-fdbff1b267a2', '80ffb91b-d13a-43f1-9f63-789f5bdb44de', 'cd07d6ef-2ee0-48e2-8b7b-f116708e07c4', 'b8668dd4-588f-48f3-ac15-3d2b0e